# Encoder-Only Transformer

- Creating a BERT Model the Easy Way
- Creating a BERT Model from Scratch with PyTorch
- Pre-training the BERT Model

**References**: [https://machinelearningmastery.com](https://machinelearningmastery.com/pretrain-a-bert-model-from-scratch/)

# 📌 STEP 1 — Install required libraries

In [2]:
# !pip install transformers

# 📌 STEP 2 — Import libraries

In [3]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
import numpy as np
from sklearn.metrics import accuracy_score

# 📌 STEP 3 — Load a small sentiment dataset

In [4]:
ds = load_dataset("imdb", split="train[:20]")
ds.column_names

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

['text', 'label']

# 📌 STEP 4 — Load BERT tokenizer

In [5]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

# 📌 STEP 5 — Tokenize the dataset

In [6]:
def tokenize(batch):
  return tokenizer(batch['text'], return_tensors="pt", max_length=512, padding=True, truncation=True)

tokenized_ds = ds.map(tokenize, batched=True)
tokenized_ds = tokenized_ds.rename_column("label", "labels")
tokenized_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [7]:
tokenized_ds.column_names

['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask']

# 📌 STEP 6 — Load pretrained BERT sentiment model



In [8]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# 📌 STEP 7 — Training arguments

In [9]:
training_args = TrainingArguments(
    output_dir="/root/my_bert_model_checkpoints",
    save_strategy="epoch",
    eval_strategy="epoch",
    learning_rate=1.5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none"
)

# 📌 STEP 8 — Build Trainer

In [10]:
def compute_metrics(p):
    predictions = np.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, predictions)}

split_datasets = tokenized_ds.train_test_split(test_size=0.2, seed=42)
train_dataset = split_datasets["train"]
eval_dataset = split_datasets["test"]

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

# 📌 STEP 9 — Train the model

In [11]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.558342,1.000000
2,No log,0.385881,1.000000
3,No log,0.372996,1.000000


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=6, training_loss=0.5225417613983154, metrics={'train_runtime': 379.7336, 'train_samples_per_second': 0.126, 'train_steps_per_second': 0.016, 'total_flos': 12629330657280.0, 'train_loss': 0.5225417613983154, 'epoch': 3.0})

# 📌 STEP 10 — Try the model on custom sentences

In [13]:
def predict_sentiment(text):
    tokens = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**tokens)
        probs = torch.softmax(outputs.logits, dim=1)
        if torch.argmax(probs) == 1:
            return "Positive 🎉", float(probs[0][1])
        else:
            return "Negative 😢", float(probs[0][0])

print(predict_sentiment("I really loved this movie!"))
print(predict_sentiment("This is the worst thing I've ever watched."))

('Negative 😢', 0.5436368584632874)
('Negative 😢', 0.5431845188140869)
